# Tutorial 12: Basic Autoencoders (PyTorch)
Converted from the TensorFlow/Keras tutorial. Covers:
1. Loading MNIST and normalizing to [0, 1]
2. Baseline convolutional autoencoder
3. Training and visualizing reconstructions
4. **Task: Improve the model** (deeper conv autoencoder with BatchNorm and strided convs), compared against the baseline.

## 1. Imports and device

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import DataLoader
from torchvision import datasets, transforms
import matplotlib.pyplot as plt

torch.manual_seed(42)
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Using device:", device)

## 2. Load MNIST
Images are scaled to [0, 1] by `ToTensor()` and have shape `(1, 28, 28)` (channel-first, PyTorch convention).

In [ ]:
transform = transforms.ToTensor()
train_set = datasets.MNIST(root="./data", train=True, download=True, transform=transform)
test_set = datasets.MNIST(root="./data", train=False, download=True, transform=transform)

train_loader = DataLoader(train_set, batch_size=128, shuffle=True)
test_loader = DataLoader(test_set, batch_size=128, shuffle=False)

x_test = torch.stack([test_set[i][0] for i in range(len(test_set))])
print("Train:", len(train_set), "Test:", len(test_set), "Shape:", tuple(train_set[0][0].shape))
print("Pixel range:", float(x_test.min()), "to", float(x_test.max()))

## 3. Baseline autoencoder
Same structure as the tutorial:
- **Encoder:** two Conv+MaxPool blocks, then a dense layer to the latent vector
- **Decoder:** dense layer back to a feature map, then two Conv+Upsample blocks and a sigmoid Conv output (28x28)

In [ ]:
class BasicAutoencoder(nn.Module):
    def __init__(self, latent_dim=64):
        super().__init__()
        # Encoder
        self.enc_conv = nn.Sequential(
            nn.Conv2d(1, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                      # 28 -> 14
            nn.Conv2d(32, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.MaxPool2d(2),                      # 14 -> 7
        )
        self.enc_dense = nn.Linear(64 * 7 * 7, latent_dim)   # latent space

        # Decoder
        self.dec_dense = nn.Linear(latent_dim, 64 * 7 * 7)
        self.dec_conv = nn.Sequential(
            nn.Conv2d(64, 64, kernel_size=3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2),          # 7 -> 14
            nn.Conv2d(64, 32, kernel_size=3, padding=1), nn.ReLU(),
            nn.Upsample(scale_factor=2),          # 14 -> 28
            nn.Conv2d(32, 1, kernel_size=3, padding=1), nn.Sigmoid(),
        )

    def encode(self, x):
        return self.enc_dense(self.enc_conv(x).flatten(1))

    def decode(self, z):
        h = self.dec_dense(z).view(-1, 64, 7, 7)
        return self.dec_conv(h)

    def forward(self, x):
        return self.decode(self.encode(x))

baseline = BasicAutoencoder().to(device)
print(baseline)
print("Parameters:", sum(p.numel() for p in baseline.parameters()))

## 4. Training utilities
Loss is binary cross-entropy (matches the sigmoid output and [0, 1] targets). The input is also the target, as in `autoencoder.fit(x_train, x_train, ...)`.

In [ ]:
def train_autoencoder(model, epochs=50, lr=1e-3, name="model"):
    opt = torch.optim.Adam(model.parameters(), lr=lr)
    history = {"train": [], "val": []}
    for epoch in range(1, epochs + 1):
        model.train()
        running = 0.0
        for x, _ in train_loader:
            x = x.to(device)
            opt.zero_grad()
            loss = F.binary_cross_entropy(model(x), x)
            loss.backward()
            opt.step()
            running += loss.item() * x.size(0)
        train_loss = running / len(train_set)

        val_loss = evaluate_loss(model)
        history["train"].append(train_loss)
        history["val"].append(val_loss)
        if epoch % 5 == 0 or epoch == 1:
            print(f"[{name}] epoch {epoch:3d}/{epochs}  train_loss={train_loss:.4f}  val_loss={val_loss:.4f}")
    return history

@torch.no_grad()
def evaluate_loss(model):
    model.eval()
    total = 0.0
    for x, _ in test_loader:
        x = x.to(device)
        total += F.binary_cross_entropy(model(x), x, reduction="sum").item()
    return total / len(test_set)

@torch.no_grad()
def reconstruction_mse(model):
    model.eval()
    recon = torch.cat([model(x.to(device)).cpu() for x, _ in test_loader])
    return F.mse_loss(recon, x_test).item()

## 5. Train the baseline
The tutorial uses `epochs=50, batch_size=128`. Lower `EPOCHS` for a quicker run.

In [ ]:
EPOCHS = 50
baseline_hist = train_autoencoder(baseline, epochs=EPOCHS, name="baseline")
print("Baseline test MSE:", round(reconstruction_mse(baseline), 5))

## 6. Visualize the results
Original test digits on top, reconstructions below.

In [ ]:
@torch.no_grad()
def show_reconstructions(model, n=10, title=""):
    model.eval()
    originals = x_test[:n].to(device)
    recons = model(originals).cpu()
    fig, axes = plt.subplots(2, n, figsize=(20, 4))
    for i in range(n):
        axes[0, i].imshow(x_test[i].squeeze(), cmap="gray"); axes[0, i].axis("off")
        axes[1, i].imshow(recons[i].squeeze(), cmap="gray"); axes[1, i].axis("off")
    axes[0, 0].set_title("Original", loc="left")
    axes[1, 0].set_title("Reconstructed", loc="left")
    fig.suptitle(title)
    plt.show()

show_reconstructions(baseline, title="Baseline autoencoder")

## 7. Task: Improve the model
Changes compared to the baseline:
- **BatchNorm + more filters** for better feature learning
- **Strided convolutions** (learned downsampling) instead of MaxPool
- **Transposed convolutions** (learned upsampling) instead of fixed Upsample
- **Convolutional bottleneck** (a 1x1 conv) that keeps spatial structure, so the latent code is 16 channels x 7 x 7 instead of one flat 64-vector
- Lightweight **dropout-free** design so training stays stable

In [ ]:
def block(in_ch, out_ch, stride=1):
    return nn.Sequential(
        nn.Conv2d(in_ch, out_ch, kernel_size=3, stride=stride, padding=1, bias=False),
        nn.BatchNorm2d(out_ch),
        nn.ReLU(inplace=True),
    )

class ImprovedConvAutoencoder(nn.Module):
    def __init__(self, latent_channels=16):
        super().__init__()
        # Encoder: 28x28x1 -> 14x14x32 -> 7x7x64 -> 7x7x16 (latent)
        self.encoder = nn.Sequential(
            block(1, 32, stride=2),       # 28 -> 14
            block(32, 64, stride=2),      # 14 -> 7
            block(64, 64),
            nn.Conv2d(64, latent_channels, kernel_size=1),   # bottleneck
        )
        # Decoder: 7x7x16 -> 7x7x64 -> 14x14x32 -> 28x28x1
        self.decoder = nn.Sequential(
            block(latent_channels, 64),
            block(64, 64),
            nn.ConvTranspose2d(64, 32, kernel_size=3, stride=2, padding=1, output_padding=1),  # 7 -> 14
            nn.BatchNorm2d(32), nn.ReLU(inplace=True),
            nn.ConvTranspose2d(32, 16, kernel_size=3, stride=2, padding=1, output_padding=1),  # 14 -> 28
            nn.BatchNorm2d(16), nn.ReLU(inplace=True),
            nn.Conv2d(16, 1, kernel_size=3, padding=1),
            nn.Sigmoid(),
        )

    def encode(self, x):
        return self.encoder(x)

    def decode(self, z):
        return self.decoder(z)

    def forward(self, x):
        return self.decode(self.encode(x))

improved = ImprovedConvAutoencoder().to(device)
print("Parameters:", sum(p.numel() for p in improved.parameters()))
print("Latent shape:", tuple(improved.encode(x_test[:1].to(device)).shape[1:]))

In [ ]:
improved_hist = train_autoencoder(improved, epochs=EPOCHS, name="improved")
print("Improved test MSE:", round(reconstruction_mse(improved), 5))

## 8. Compare the two models

In [ ]:
print(f"{'Model':<12}{'Params':>10}{'Test MSE':>12}")
for name, m in [("Baseline", baseline), ("Improved", improved)]:
    p = sum(q.numel() for q in m.parameters())
    print(f"{name:<12}{p:>10,}{reconstruction_mse(m):>12.5f}")

fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(baseline_hist["val"], label="Baseline (val BCE)")
ax.plot(improved_hist["val"], label="Improved (val BCE)")
ax.set_xlabel("Epoch"); ax.set_ylabel("Validation BCE"); ax.legend(); ax.grid(alpha=0.3)
plt.show()

show_reconstructions(improved, title="Improved autoencoder")

## 9. Exercises (optional)
- Change `latent_channels` (e.g. 4, 8, 32) and observe the trade-off between compression and reconstruction quality.
- Add denoising: corrupt inputs with noise during training but keep the clean images as targets.
- Try MSE loss instead of BCE and compare the reconstructions.